# <a id='toc1_'></a>[Gene JAR Example Use](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [Gene JAR Example Use](#toc1_)    
  - [Set up](#toc1_1_)    
  - [Lookup function](#toc1_2_)    
  - [Ambiguity Check function](#toc1_3_)    
  - [Resolve function](#toc1_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Set up](#toc0_)

In [1]:
from pathlib import Path

import pandas as pd

from gene_jar import GeneJar, MatchType, SearchColumn

In [2]:
# Define data directory
DATA_DIR = Path("/Users/rsaxs014/Desktop/gene-jar/src/gene_jar/data")

# List of (dict key, filename) pairs
files = [
    ("primary_df", "summary_df.h5"),
    ("ortholog_df", "ortholog_match_subset_genes_df.h5"),
    ("flj_clone_df", "flj_clone_symbol_match_subset_genes_df.h5"),
    ("phenotype_df", "phenotype_match_subset_genes_df.h5"),
    ("hgnc_gene_group_df", "gene_group_prefix_match_subset_genes_df.h5"),
    ("gene_id_df", "gene_id_match_subset_genes_df.h5"),
    ("mgi_withdrawn_df", "mgi_withdrawn_symbol_match_subset_genes_df.h5"),
    ("ncbi_related_gene_df", "ncbi_related_gene_match_subset_genes_df.h5"),
    (
        "ncbi_gene_interaction_df",
        "ncbi_gene_interaction_symbol_match_subset_genes_df.h5",
    ),
    ("ncbi_gene_neighbor_df", "ncbi_gene_neighbor_match_subset_genes_df.h5"),
    ("placeholder_df", "placeholder_match_subset_genes_df.h5"),
    ("previous_df", "previous_symbol_match_subset_genes_df.h5"),
    ("protein_mass_df", "protein_mass_match_subset_genes_df.h5"),
    ("alternate_abbreviation_df", "alt_abbrev_symbol_match_subset_genes_df.h5"),
]

# Load all DataFrames into a dictionary
dfs = {key: pd.read_hdf(DATA_DIR / filename, key="df") for key, filename in files}

# Create the resolver
gj = GeneJar(**dfs)

## <a id='toc1_2_'></a>[Lookup function](#toc0_)

This function allows searching for gene symbols that are primary or aliases with the option to get more information on the gene-to-symbol relationship in the qualifier column

Lookup table columns:
- *HGNC_ID* - HGNC ID associated with the gene
- *ENSG_ID* - Ensembl ID associated with the gene
- *NCBI_ID* - NCBI Gene ID associated with the gene
- *primary_gene_symbol* - primary gene symbol associated with the gene records with the referenced identifiers
- *gene_symbol* - gene symbols associated with the gene identifiers
- *captured* - "T" the gene-to-symbol relationship between "primary_gene_symbol" as the gene and "gene_symbol" has been annotated, "F" it has not
- *captured as:* - The gene-to-symbol relationship value(s)

In [3]:
# A complete identical match as a pirmary gene symbol or an alias is the default
gj.lookup("ABL")

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:
644,{HGNC:76},{ENSG00000097007},{GENE ID:25},ABL1,ABL,T,"Previous Symbol, Gene Group Symbol"
27508,{HGNC:56039},{ENSG00000261189},{GENE ID:101928076},DSP-AS1,ABL,F,NaN
76102,{HGNC:7467},{ENSG00000138823},{GENE ID:4547},MTTP,ABL,T,Phenotype Symbol


In [4]:
# Alias symbol only
gj.lookup("ABL", search_column=SearchColumn.ALIAS)

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:
644,{HGNC:76},{ENSG00000097007},{GENE ID:25},ABL1,ABL,T,"Previous Symbol, Gene Group Symbol"
27508,{HGNC:56039},{ENSG00000261189},{GENE ID:101928076},DSP-AS1,ABL,F,NaN
76102,{HGNC:7467},{ENSG00000138823},{GENE ID:4547},MTTP,ABL,T,Phenotype Symbol


In [5]:
# Primary gene symbol only
gj.lookup("ABL", search_column=SearchColumn.PRIMARY)

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:


In [6]:
# There is an option to search for a partial match that will lookup symbols that contain "abl"
gj.lookup("ABL", match_type=MatchType.PARTIAL)

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:
607,{HGNC:11320},{ENSG00000136754},{GENE ID:10006},ABI1,ABLBP4,F,NaN
618,{HGNC:24011},{ENSG00000138443},{GENE ID:10152},ABI2,ABLBP3,F,NaN
622,{HGNC:24011},{ENSG00000138443},{GENE ID:10152},ABI2,AblBP3,F,NaN
644,{HGNC:76},{ENSG00000097007},{GENE ID:25},ABL1,ABL,T,"Previous Symbol, Gene Group Symbol"
645,{HGNC:76},{ENSG00000097007},{GENE ID:25},ABL1,ABL1,T,Primary Gene Symbol
...,...,...,...,...,...,...,...
96515,{HGNC:24703},{ENSG00000196642},{GENE ID:55684},RABL6,Parf,F,NaN
96516,{HGNC:24703},{ENSG00000196642},{GENE ID:55684},RABL6,RABL6,T,Primary Gene Symbol
96517,{HGNC:24703},{ENSG00000196642},{GENE ID:55684},RABL6,RBEL1,T,Alternate Abbreviation Symbol
96518,{HGNC:24703},{ENSG00000196642},{GENE ID:55684},RABL6,bA216L13.9,F,NaN


In [7]:
# To get more information on the qualifier of the relationship, include the symbol category parameter
gj.lookup("ABL", match_type=MatchType.PARTIAL, symbol_category="Gene Group Symbol")

,primary_gene_symbol,alias_symbol,HGNC_ID,ENSG_ID,NCBI_ID,Prefix Gene Group Symbol Match,Matching Abbreviation
546,ABL1,ABL,{HGNC:76},{ENSG00000097007},{GENE ID:25},True,ABL
557,ABL2,ABLL,{HGNC:77},{ENSG00000143322},{GENE ID:27},True,ABL
40424,IFT22,RABL5,{HGNC:21895},{ENSG00000128581},{GENE ID:64792},True,RABL
40435,IFT27,RABL4,{HGNC:18626},{ENSG00000100360},{GENE ID:11020},True,RABL
82694,RAB5C,RABL,{HGNC:9785},{ENSG00000108774},{GENE ID:5878},True,RAB


## <a id='toc1_3_'></a>[Ambiguity Check function](#toc0_)

This function identifies ambiguous gene symbols

In [8]:
# This will check if the query is associated with more than one gene concept
result = gj.ambiguity_check("ABL")

# The ambiguity status is stored in the `is_ambiguous` property of the returned `AmbiguityResult` object.
result.is_ambiguous

True

In [9]:
# The returned `AmbiguityResult` object stores the ambiguity status and a list of matching gene concepts with their associated identifiers.
result

AmbiguityResult(is_ambiguous=True, gene_matches=[GeneMatch(primary_gene_symbol='ABL1', hgnc_id={'HGNC:76'}, ncbi_id={'GENE ID:25'}, ensg_id={'ENSG00000097007'}), GeneMatch(primary_gene_symbol='MTTP', hgnc_id={'HGNC:7467'}, ncbi_id={'GENE ID:4547'}, ensg_id={'ENSG00000138823'})])

In [10]:
# The `gene_matches` attribute contains the gene concepts associated with the queried symbol and their corresponding identifiers.
result.gene_matches

[GeneMatch(primary_gene_symbol='ABL1', hgnc_id={'HGNC:76'}, ncbi_id={'GENE ID:25'}, ensg_id={'ENSG00000097007'}),
 GeneMatch(primary_gene_symbol='MTTP', hgnc_id={'HGNC:7467'}, ncbi_id={'GENE ID:4547'}, ensg_id={'ENSG00000138823'})]

In [11]:
# Individual gene matches can be accessed by index, and the `primary_gene_symbol` attribute contains the primary symbol for the matched gene concept.
result.gene_matches[0].primary_gene_symbol

'ABL1'

In [12]:
# The `hgnc_id` attribute contains the HGNC identifier(s) associated with the matched gene concept.
result.gene_matches[0].hgnc_id

{'HGNC:76'}

In [13]:
result.gene_matches[0].ncbi_id

{'GENE ID:25'}

In [14]:
result.gene_matches[0].ensg_id

{'ENSG00000097007'}

## <a id='toc1_4_'></a>[Resolve function](#toc0_)

This function ranks genes associated to ambiguous gene symbols based on gene symbol categories

Resolve table columns:
- *candidate_rank** - Rank of the candidate gene based on the priority of its highest-ranked matched relationship category and the number of matched relationship categories
- *primary_gene_symbol** - Primary gene symbol associated with the candidate gene
- *HGNC_ID* - HGNC ID(s) associated with the candidate gene
- *NCBI_ID* - NCBI Gene ID(s) associated with the candidate gene
- *ENSG_ID* - Ensembl ID(s) associated with the candidate gene
- *best_category* - Highest-ranked gene symbol relationship category matched for the candidate gene
- *qualifier* - Additional information describing the relationship represented by the `best_category`, when available
- *matched_symbol* - Gene symbol(s) that matched the queried symbol
- *matched_categories* - Gene symbol relationship category or categories in which the queried symbol matched the candidate gene
- *relationship_count* - Number of gene symbol relationship categories in which the queried symbol matched the candidate gene

In [15]:
# Resolve the queried symbol and return ranked candidate gene concepts based on the identified symbol relationships. Identical symbol matching is used by default.
gj.resolve("KRAS")

,candidate_rank,primary_gene_symbol,HGNC_ID,NCBI_ID,ENSG_ID,best_category,qualifier,matched_symbol,matched_categories,relationship_count
0,1,KRAS,{HGNC:6407},{GENE ID:3845},{ENSG00000133703},Primary Gene Symbol,{},{KRAS},[Primary Gene Symbol],1
1,2,NRAS,{HGNC:7989},{GENE ID:4893},{ENSG00000213281},Ortholog Symbol,{'Matching Species': ['Norway rat - BN/NHsdMcw...,{KRAS},[Ortholog Symbol],1


In [16]:
# Partial matching can be specified with `MatchType.PARTIAL` to return candidate gene concepts containing the queried symbol.
gj.resolve(symbol="KRAS", match_type=MatchType.PARTIAL)

,candidate_rank,primary_gene_symbol,HGNC_ID,NCBI_ID,ENSG_ID,best_category,qualifier,matched_symbol,matched_categories,relationship_count
0,1,KRAS,"{HGNC:6407, {'HGNC:6407'}}","{{'GENE ID:3845'}, GENE ID:3845}","{{'ENSG00000133703'}, ENSG00000133703}",Primary Gene Symbol,{},"{KRAS, KRAS2}","[Primary Gene Symbol, Previous Symbol, Alterna...",4
1,2,KRASP1,"{HGNC:6406, {'HGNC:6406'}}","{GENE ID:3844, {'GENE ID:3844'}}","{{'ENSG00000220635'}, set(), ENSG00000220635}",Primary Gene Symbol,{},"{KRASP1, KRAS1P, c-Kras1}","[Primary Gene Symbol, Previous Symbol, Alterna...",3
2,3,NRAS,{HGNC:7989},{GENE ID:4893},{ENSG00000213281},Ortholog Symbol,{'Matching Species': ['Norway rat - BN/NHsdMcw...,{KRAS},[Ortholog Symbol],1


In [17]:
gj.resolve("ABL")

,candidate_rank,primary_gene_symbol,HGNC_ID,NCBI_ID,ENSG_ID,best_category,qualifier,matched_symbol,matched_categories,relationship_count
0,1,ABL1,{HGNC:76},{GENE ID:25},{ENSG00000097007},Previous Symbol,{'Previous Symbol Source': 'HGNC'},{ABL},"[Previous Symbol, Gene Group Symbol]",2
1,2,MTTP,{HGNC:7467},{GENE ID:4547},{ENSG00000138823},Phenotype Symbol,{'Matching Phenotype Symbol': 'ABL'},{ABL},[Phenotype Symbol],1


In [18]:
gj.resolve(symbol="ABL", match_type=MatchType.PARTIAL)

,candidate_rank,primary_gene_symbol,HGNC_ID,NCBI_ID,ENSG_ID,best_category,qualifier,matched_symbol,matched_categories,relationship_count
0,1,ABL1,"{HGNC:76, {'HGNC:76'}}","{GENE ID:25, {'GENE ID:25'}}","{{'ENSG00000097007'}, ENSG00000097007}",Primary Gene Symbol,{},"{ABL, c-ABL, C-ABL, c-ABL1, ABL1}","[Primary Gene Symbol, Previous Symbol, Alterna...",4
1,2,ABL2,{HGNC:77},{GENE ID:27},{ENSG00000143322},Primary Gene Symbol,{},"{ABL2, ABLL}","[Primary Gene Symbol, Previous Symbol, Withdra...",4
2,3,ABLIM1,"{{'HGNC:78'}, HGNC:78}","{{'GENE ID:3983'}, GENE ID:3983}","{{'ENSG00000099204'}, ENSG00000099204}",Primary Gene Symbol,{},"{ABLIM, abLIM, abLIM-1, ABLIM1}","[Primary Gene Symbol, Previous Symbol, Alterna...",3
3,4,CABLES1,"{HGNC:25097, {'HGNC:25097'}}","{GENE ID:91768, {'GENE ID:91768'}}","{ENSG00000134508, {'ENSG00000134508'}}",Primary Gene Symbol,{},"{CABL1, CABLES1}","[Primary Gene Symbol, Alternate Abbreviation S...",2
4,5,DIABLO,"{HGNC:21528, {'HGNC:21528'}}","{GENE ID:56616, {'GENE ID:56616'}}","{{'ENSG00000184047'}, ENSG00000184047}",Primary Gene Symbol,{},"{DIABLO-S, DIABLO}","[Primary Gene Symbol, Alternate Abbreviation S...",2
5,6,ABLIM2,{HGNC:19195},{GENE ID:84448},{ENSG00000163995},Primary Gene Symbol,{},{ABLIM2},[Primary Gene Symbol],1
6,7,ABLIM3,{HGNC:29132},{GENE ID:22885},{ENSG00000173210},Primary Gene Symbol,{},{ABLIM3},[Primary Gene Symbol],1
7,8,CABLES2,{HGNC:16143},{GENE ID:81928},{ENSG00000149679},Primary Gene Symbol,{},{CABLES2},[Primary Gene Symbol],1
8,9,ERVPABLB-1,{HGNC:39042},{GENE ID:100862693},{},Primary Gene Symbol,{},{ERVPABLB-1},[Primary Gene Symbol],1
9,10,RABL2A,{HGNC:9799},{GENE ID:11159},{ENSG00000144134},Primary Gene Symbol,{},{RABL2A},[Primary Gene Symbol],1


In [19]:
gj.resolve("DBI")

,candidate_rank,primary_gene_symbol,HGNC_ID,NCBI_ID,ENSG_ID,best_category,qualifier,matched_symbol,matched_categories,relationship_count
0,1,DBI,{HGNC:2690},{GENE ID:1622},{ENSG00000155368},Primary Gene Symbol,{},{DBI},[Primary Gene Symbol],1
1,2,TSPO,{HGNC:1158},{706},{ENSG00000100300},Gene Interaction Symbol,{},{DBI},[Gene Interaction Symbol],1


In [20]:
gj.resolve(symbol="DBI", match_type=MatchType.PARTIAL)

,candidate_rank,primary_gene_symbol,HGNC_ID,NCBI_ID,ENSG_ID,best_category,qualifier,matched_symbol,matched_categories,relationship_count
0,1,DBIP1,{HGNC:2692},{GENE ID:1624},{ENSG00000218748},Primary Gene Symbol,{},"{DBIL2, DBIP1}","[Primary Gene Symbol, Previous Symbol]",2
1,2,DBIP2,{HGNC:2691},{GENE ID:1623},{ENSG00000251366},Primary Gene Symbol,{},"{DBIL1, DBIP2}","[Primary Gene Symbol, Previous Symbol]",2
2,3,DBI,{HGNC:2690},{GENE ID:1622},{ENSG00000155368},Primary Gene Symbol,{},{DBI},[Primary Gene Symbol],1
3,4,DBIL5P,{HGNC:38519},{GENE ID:100131454},"{ENSG00000293440, ENSG00000231784}",Primary Gene Symbol,{},{DBIL5P},[Primary Gene Symbol],1
4,5,DBIL5P2,{HGNC:38518},{GENE ID:100169989},"{ENSG00000293077, ENSG00000242412}",Primary Gene Symbol,{},{DBIL5P2},[Primary Gene Symbol],1
5,6,DBIP3,{HGNC:38517},{GENE ID:791082},{},Primary Gene Symbol,{},{DBIP3},[Primary Gene Symbol],1
6,7,TSPO,{HGNC:1158},{706},{ENSG00000100300},Gene Interaction Symbol,{},{DBI},[Gene Interaction Symbol],1
